# xLSTM-GRPO 5단계 분할 매매 & 1~5초 패턴 훈련 (Google Colab GPU)

이 노트북은 `train_xlstm.py`를 사용하여 5단계 분할 매매 및 1~5초 급등 패턴 탐지/자동 손절 기반 강화학습 스캘핑 봇을 Colab GPU에서 훈련합니다.

**훈련 특징:**
- 5단계 분할 매수/매도 (`MAX_STAGES = 5`), 단계별 독립 진입가 정산
- 1~5초 가격 미상승 시 즉시 자동 손절 (-1.0 페널티)
- 1~5초 가격 상승 포착 시 패턴 탐지 보너스 (+0.5)
- `--seq_len 1024`, `--features 28` (관측차원 43)

**사전 준비:**
1. Colab 왼쪽 자물쇠 아이콘 → Secret 추가
   - `GITHUB_TOKEN`: GitHub Personal Access Token (private repo 접근용)
2. Google Drive에 추출 데이터셋 업로드:
   - `MyDrive/ColabData/stockbot/models/grpo_xlstm_v1/extracted_episodes`

## 1. GPU 확인

In [ ]:
import torch
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem:.1f} GB")
else:
    print("⚠️ GPU를 사용할 수 없습니다. 런타임 → 런타임 유형 변경 → GPU 선택 후 재시작하세요.")

## 2. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 필수 패키지 설치

In [ ]:
!pip install -q duckdb pandas pyarrow python-dotenv stable-baselines3 tensorboard

## 4. 프로젝트 코드 준비 (GitHub Clone)
Colab Secret에 `GITHUB_TOKEN`이 설정되어 있으면 private repo를 자동으로 클론합니다.

In [ ]:
import os
import sys
from google.colab import userdata

repo_path = '/content/stock-bot2'
REVISION = None # @param {type:"string"}
# 원하는 revision (branch, tag, commit hash)을 여기에 입력하세요. 비워두면 기본 브랜치를 사용합니다.

try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 GitHub Token 확인됨")
except:
    use_token = False
    print("⚠️ GITHUB_TOKEN 없음. public 클론 시도")

if not os.path.exists(repo_path):
    print("📥 저장소 클론 중...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git {repo_path}
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git {repo_path}
    print("✅ 클론 완료!")

    %cd {repo_path}
    if REVISION:
        print(f"⚙️ {REVISION} revision 체크아웃 중...")
        !git checkout {REVISION}
        print("✅ 체크아웃 완료!")

else:
    print("📁 저장소 이미 존재 → 업데이트 중...")
    %cd {repo_path}
    !git fetch origin dev-tf-rl
    if REVISION:
        print(f"⚙️ {REVISION} revision 체크아웃 중...")
        !git checkout {REVISION}
        print("✅ 체크아웃 완료!")
    else:
        print("⚙️ 기본 브랜치 업데이트 중...")
        !git pull
        print("✅ 업데이트 완료!")

%cd {repo_path}
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print(f"📂 작업 디렉토리: {os.getcwd()}")

## 5. 추출 데이터 준비 (Drive → Local 복사)
Google Drive에 저장된 extracted_episodes 캐시 데이터를 Colab 로컬(`/content/extracted_episodes`)로 복사합니다.

In [ ]:
!cp -r /content/drive/MyDrive/ColabData/datasets/stockbot/20260811/extracted_episodes /content/

## 6. 훈련 출력 디렉토리 설정
체크포인트와 로그를 Google Drive에 저장하여 Colab 세션이 종료되어도 유실되지 않도록 합니다.

In [ ]:
# ⚙️ 필요에 따라 경로를 수정하세요
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ColabData/stockbot/models/grpo_xlstm_v4'

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT_DIR}/checkpoints', exist_ok=True)
print(f"✅ 출력 디렉토리: {DRIVE_OUTPUT_DIR}")

## 7. xLSTM-GRPO 훈련 시작
처음 훈련하는 경우 `LOAD_POLICY = None`으로 설정하세요.
재개하는 경우 `checkpoint_iter<N>.pt` 또는 `checkpoint_best.pt` 경로를 지정하세요.

In [ ]:
# ── 설정 ─────────────────────────────────────────
OUTPUT_DIR     = DRIVE_OUTPUT_DIR
vram_preset    = 'large' # 'small', 'medium', 'large', 'xlarge'
batch_size     = 256
kl_target      = 0.05
clip           = 0.2
max_trades_per_episode = 2
total_timesteps = 1000000
seq_len        = 1024
features       = 27
num_workers    = 16
entropy_coef   = 0.02 # 0.005
transaction_cost = 0.00015
buy_tax         = 0.0
sell_tax        = 0.0018
no_trade_penalty = 0.5 # 0.0
step_reward_scale = 1.0
revert_patience = 30
buy_signal_bonus = 0.5
win_bonus      = 2.5 # 1.5
loss_penalty   = 1.5
LR             = 3e-5

LOAD_POLICY    = f'{DRIVE_OUTPUT_DIR}/checkpoints/checkpoint_best.pt'  # 신규 훈련 시 None으로 지정
# ─────────────────────────────────────────────────

load_policy_arg = f'--load_policy "{LOAD_POLICY}"' if LOAD_POLICY and os.path.exists(LOAD_POLICY) else ''

cmd = f'''\
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python ai_trader/grpo/train_xlstm.py \
    {load_policy_arg} \
    --extracted_dir "/content/extracted_episodes" \
    --episodes_per_group 8 --num_groups 16 \
    --checkpoint_segments 8 \
    --use_gae True \
    --vram_preset {vram_preset} \
    --batch_size {batch_size} \
    --kl_target {kl_target} \
    --clip {clip} \
    --max_trades_per_episode {max_trades_per_episode} \
    --seq_len {seq_len} \
    --features {features} \
    --total_timesteps {total_timesteps} \
    --output_dir "{OUTPUT_DIR}" \
    --num_workers {num_workers} \
    --entropy_coef {entropy_coef} \
    --transaction_cost {transaction_cost} \
    --buy_tax {buy_tax} \
    --sell_tax {sell_tax} \
    --no_trade_penalty {no_trade_penalty} \
    --step_reward_scale {step_reward_scale} \
    --revert_patience {revert_patience} \
    --buy_signal_bonus {buy_signal_bonus} \
    --win_bonus {win_bonus} \
    --loss_penalty {loss_penalty} \
    --lr {LR}
'''

print("🚀 훈련 시작...")
print(f"명령어:\n{cmd}")
!{cmd}


## 8. 훈련 모니터링 (TensorBoard)

In [ ]:
%load_ext tensorboard
TENSORBOARD_LOG_DIR = f'{DRIVE_OUTPUT_DIR}/tensorboard_logs'
%tensorboard --logdir {TENSORBOARD_LOG_DIR}

## 9. 로그 분석 (선택)
훈련 중 또는 후에 실행하여 지표를 요약합니다.

In [ ]:
import re
import glob

def parse_log(filepath):
    pattern = re.compile(
        r"Iteration (\d+)/.*?Mean Reward: ([\-\d.]+) \| Win Rate: ([\d.]+)% \| "
        r"Trades: (\d+) \| Sharpe: ([\-\d.]+) \| AvgHold: ([\d.]+)s"
    )
    metrics = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                metrics.append({
                    'iter': int(m.group(1)), 'reward': float(m.group(2)),
                    'win_rate': float(m.group(3)), 'trades': int(m.group(4)),
                    'sharpe': float(m.group(5)), 'avg_hold': float(m.group(6))
                })
    return metrics

log_files = sorted(glob.glob('logs/train_xlstm_*.log'), reverse=True)
if log_files:
    latest = log_files[0]
    print(f"📋 최신 로그: {latest}")
    data = parse_log(latest)
    if data:
        print(f"\n총 {len(data)} iterations 파싱 완료 ({data[0]['iter']} → {data[-1]['iter']})")
        print(f"\n최근 100 iteration 평균:")
        recent = data[-100:]
        print(f"  Mean Reward : {sum(x['reward'] for x in recent)/len(recent):.4f}")
        print(f"  Win Rate    : {sum(x['win_rate'] for x in recent)/len(recent):.2f}%")
        print(f"  Avg Trades  : {sum(x['trades'] for x in recent)/len(recent):.1f}")
        print(f"  Avg Sharpe  : {sum(x['sharpe'] for x in recent)/len(recent):.3f}")
        print(f"  Avg Hold    : {sum(x['avg_hold'] for x in recent)/len(recent):.2f}s")
    else:
        print("❌ 아직 iteration 로그가 없습니다. 훈련을 먼저 시작하세요.")
else:
    print("❌ 로그 파일이 없습니다.")